# 6. CREDIT APPROVAL (CRX) - Preprocesamiento
**Tipo:** CLASIFICACIÓN BINARIA (aprobar/rechazar crédito)
**Variable objetivo:** A16 (+ = Aprobado = 1, - = Rechazado = 0)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ============================================================
# CARGAR DATOS (UCI - sin header, datos ofuscados)
# ============================================================
columnas = ['A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'A7', 'A8', 'A9', 
            'A10', 'A11', 'A12', 'A13', 'A14', 'A15', 'A16']

df = pd.read_csv('crx.data', names=columnas, na_values='?')
print(f"Dimensiones: {df.shape}")
print(f"Columnas: {list(df.columns)}")
df.head()

In [ ]:
# ============================================================
# EXPLORAR TIPOS DE DATOS
# ============================================================
print("Tipos de datos:")
print(df.dtypes)
print(f"\nNulos por columna:")
print(df.isnull().sum())

In [ ]:
# ============================================================
# LIMPIEZA DE NULOS (Imputación)
# ============================================================
# Numéricas: rellenar con mediana
# Categóricas: rellenar con moda

for col in df.columns:
    if df[col].isnull().sum() > 0:
        if df[col].dtype == 'object':
            df[col] = df[col].fillna(df[col].mode()[0])
        else:
            df[col] = df[col].fillna(df[col].median())

print(f"Nulos restantes: {df.isnull().sum().sum()}")

In [ ]:
# ============================================================
# PREPARAR VARIABLE OBJETIVO
# ============================================================
# A16: + = Aprobado (1), - = Rechazado (0)
df['A16'] = df['A16'].map({'+': 1, '-': 0})
y = df['A16'].values

print(f"Balance de clases: {np.bincount(y)}")
print(f"Proporción aprobados: {y.mean():.2%}")

In [ ]:
# ============================================================
# PREPARAR CARACTERÍSTICAS
# ============================================================
X_df = df.drop(columns=['A16'])

# Eliminar A14 (código postal) - muchos valores únicos, genera ruido
if 'A14' in X_df.columns:
    X_df = X_df.drop(columns=['A14'])

print(f"Columnas: {list(X_df.columns)}")

In [ ]:
# ============================================================
# ENCODING DE VARIABLES
# ============================================================
# Convertir columnas binarias (t/f) a 1/0
binary_cols = ['A9', 'A10', 'A12']  # t=True, f=False
for col in binary_cols:
    if col in X_df.columns:
        X_df[col] = X_df[col].map({'t': 1, 'f': 0})

# Identificar categóricas restantes
cat_cols = X_df.select_dtypes(include=['object']).columns.tolist()
print(f"Categóricas para One-Hot: {cat_cols}")

# One-Hot Encoding
X_encoded = pd.get_dummies(X_df, columns=cat_cols, drop_first=True)
print(f"\nDimensiones después de encoding: {X_encoded.shape}")

In [ ]:
# ============================================================
# DIVISIÓN DE DATOS
# ============================================================
X = X_encoded.to_numpy().astype(float)  # Asegurar tipo float

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.333, random_state=42)

print(f"Train: {X_train.shape[0]}")
print(f"Val: {X_val.shape[0]}")
print(f"Test: {X_test.shape[0]}")

In [ ]:
# ============================================================
# NORMALIZACIÓN
# ============================================================
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

print("✅ Datos listos para CLASIFICACIÓN BINARIA")
print(f"Características: {X_train.shape[1]}")

## MÉTODO TRADICIONAL

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

modelo = LogisticRegression(max_iter=1000)
modelo.fit(X_train, y_train)
y_pred = modelo.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred))

## CON PYTORCH

In [ ]:
import torch
import torch.nn as nn

X_train_t = torch.FloatTensor(X_train)
y_train_t = torch.FloatTensor(y_train).reshape(-1, 1)

class ClasificadorCredito(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.red = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.red(x)

modelo = ClasificadorCredito(X_train_t.shape[1])
criterio = nn.BCELoss()
optimizador = torch.optim.Adam(modelo.parameters(), lr=0.001)
print(modelo)